In [1]:
# ================================
# Stochastic SQP-PINN (JAX) - POINTWISE (SIGNED) constraints
# HOLD base samples (data/obj/con) + JITTER every iteration
#
# WHAT YOU ASKED:
#   ✅ Constraint scaling: scalar RMS(J) + EMA for first 100 iterations, then FREEZE
#   ✅ Scale BOTH c and J:  Cons = s*c_un,  Jac = s*J_un
#   ✅ "shrink-only" scaling: s = target / max(target, ema_mag)  => s <= 1
#   ✅ Do NOT scale objective: scale_obj = 1.0
#
# NOTE:
#   - Your KKT LM invariance is kept: lam_eff = lam*(s_con^2)
# ================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time, math
from functools import partial
import numpy as np
import scipy.io

import jax
jax.config.update("jax_default_matmul_precision", "tensorfloat32")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian

# -----------------------------
# Precision
# -----------------------------
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = 1e-12

# -----------------------------
# Batch sizes / region structure
# -----------------------------
B_DATA = 0  # set >0 if you want data objective too

# PDE cells (constraints)
NX_PDE_MOM = 30
NT_PDE_MOM = 30
K_PDE = NX_PDE_MOM * NT_PDE_MOM
N_PDE_PER_CELL = 1
B_F_TOTAL = K_PDE * N_PDE_PER_CELL

# BC bins (constraints)
K_BC = 100
N_BC_PER_BIN = 1
B_BC_TOTAL = K_BC * N_BC_PER_BIN

# IC bins (constraints)
K_IC = 200
N_IC_PER_BIN = 1
B_IC_TOTAL = K_IC * N_IC_PER_BIN

M_CON = K_PDE + 2 * K_BC + K_IC

# PDE objective sampling (separate from constraints)
NX_PDE_OBJ = 80
NT_PDE_OBJ = 80
K_PDE_OBJ = NX_PDE_OBJ * NT_PDE_OBJ  # 6400 base points (1 per cell)

# -----------------------------
# 1) MLP
# -----------------------------
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h  # (N,1)

# -----------------------------
# 2) Flatten/unflatten
# -----------------------------
def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)

def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape); idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape); idx += b_size
        params.append({"W": W, "b": b})
    return params

# -----------------------------
# 3) Load burgers.mat
# -----------------------------
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)
    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]  # (nt, nx)
    nu = float(np.array(d["nu"]).squeeze())
    return t, x, usol, nu

def build_full_data_points(t_np, x_np, usol_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_data = np.stack([X.reshape(-1), T.reshape(-1)], 1)
    u_data = usol_np.reshape(-1)
    return X_data, u_data

# -----------------------------
# 4) Sampling utilities
# -----------------------------
def sample_data_batch(key, X_data, u_data, B):
    N = X_data.shape[0]
    idx = random.randint(key, (B,), 0, N)
    return X_data[idx], u_data[idx]

def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)

def sample_pde_obj_base_stratified(key, x_min, x_max, t_min, t_max):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(NX_PDE_OBJ)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(NT_PDE_OBJ)

    it, ix = jnp.meshgrid(
        jnp.arange(NT_PDE_OBJ),
        jnp.arange(NX_PDE_OBJ),
        indexing="ij"
    )
    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K_PDE_OBJ, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt
    return jnp.stack([xs, ts], axis=1)  # (K_PDE_OBJ, 2)

def sample_pde_stratified(key, x_min, x_max, t_min, t_max):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(NX_PDE_MOM)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(NT_PDE_MOM)

    it, ix = jnp.meshgrid(
        jnp.arange(NT_PDE_MOM),
        jnp.arange(NX_PDE_MOM),
        indexing="ij"
    )
    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids_cell = (it * NX_PDE_MOM + ix).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K_PDE, N_PDE_PER_CELL, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0[:, None] + u[:, :, 0] * dx
    ts = t0[:, None] + u[:, :, 1] * dt

    X_f = jnp.stack([xs, ts], axis=-1).reshape(-1, 2)
    ids = jnp.repeat(ids_cell, N_PDE_PER_CELL)
    return X_f, ids

def sample_bc_stratified(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)
    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0[:, None] + u * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)
    return XL, XR, ids

def sample_ic_stratified(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)
    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC, N_IC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0[:, None] + u * dx).reshape(-1, 1)
    ids = jnp.repeat(j, N_IC_PER_BIN)

    Xic = jnp.concatenate([xs, DTYPE(t0) * jnp.ones_like(xs)], axis=1)
    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)
    return Xic, u0, ids

# -----------------------------
# 4.5) Jitter (x,t) -> (x+noise, t+noise)
# -----------------------------
def jitter_points(key, X, x_min, x_max, t_min, t_max,
                  sigma_x, sigma_t,
                  kind="gaussian",
                  clip_k=None):
    if (sigma_x <= 0.0) and (sigma_t <= 0.0):
        return X

    key, kx, kt = random.split(key, 3)

    if kind == "gaussian":
        dx = DTYPE(sigma_x) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
        dt = DTYPE(sigma_t) * random.normal(kt, (X.shape[0],), dtype=DTYPE)
        if clip_k is not None:
            k = DTYPE(clip_k)
            if sigma_x > 0:
                dx = jnp.clip(dx, -k * DTYPE(sigma_x), +k * DTYPE(sigma_x))
            if sigma_t > 0:
                dt = jnp.clip(dt, -k * DTYPE(sigma_t), +k * DTYPE(sigma_t))
    elif kind == "uniform":
        dx = DTYPE(sigma_x) * (2.0 * random.uniform(kx, (X.shape[0],), dtype=DTYPE) - 1.0)
        dt = DTYPE(sigma_t) * (2.0 * random.uniform(kt, (X.shape[0],), dtype=DTYPE) - 1.0)
    else:
        raise ValueError("kind must be 'gaussian' or 'uniform'")

    x = jnp.clip(X[:, 0] + dx, DTYPE(x_min), DTYPE(x_max))
    t = jnp.clip(X[:, 1] + dt, DTYPE(t_min), DTYPE(t_max))
    return jnp.stack([x, t], axis=1)

# -----------------------------
# 5) PDE residual
# -----------------------------
def pde_residual_unscaled(params, X_f, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H = vmap(hessian(u_fun))(X_f)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx

def pde_mse_on_points(theta, shapes, X_pts, nu):
    params = unflatten_params(theta, shapes)
    r = pde_residual_unscaled(params, X_pts, nu)
    return jnp.mean(r**2)

# -----------------------------
# 6) Objective (NO scaling)
# -----------------------------
def data_mse_unscaled(params, Xb, ub):
    pred = mlp_apply(params, Xb)[:, 0]
    return jnp.mean((pred - ub) ** 2)

@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_data_scaled_batch(theta, shapes, Xb, ub, scale_obj):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        return data_mse_unscaled(params, Xb, ub)
    val, g = jax.value_and_grad(obj_theta)(theta)
    return scale_obj * val, scale_obj * g

@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_pde_obj(theta, shapes, X_obj, nu, w_pde):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        r = pde_residual_unscaled(params, X_obj, nu)
        return DTYPE(w_pde) * jnp.mean(r**2)
    val, g = jax.value_and_grad(obj_theta)(theta)
    return val, g

# -----------------------------
# 7) BC helper u, ux
# -----------------------------
@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]
    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux

# -----------------------------
# 8) POINTWISE/SIGNED constraints + Jacobian
# -----------------------------
@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector_cells_unscaled(theta, shapes,
                                     X_f, ids_f,
                                     X_bc_L, X_bc_R, ids_bc,
                                     X_ic, u0_ic, ids_ic,
                                     nu):
    params = unflatten_params(theta, shapes)

    # PDE signed residual
    r = pde_residual_unscaled(params, X_f, nu)
    c_pde = segment_sum(r, ids_f, K_PDE) / DTYPE(N_PDE_PER_CELL)

    # BC signed differences
    uL, uxL = u_and_ux(params, X_bc_L)
    uR, uxR = u_and_ux(params, X_bc_R)
    dv = (uL - uR)
    dd = (uxL - uxR)
    c_bc_val = segment_sum(dv, ids_bc, K_BC) / DTYPE(N_BC_PER_BIN)
    c_bc_der = segment_sum(dd, ids_bc, K_BC) / DTYPE(N_BC_PER_BIN)

    # IC signed mismatch
    u_ic = mlp_apply(params, X_ic)[:, 0]
    di = (u_ic - u0_ic)
    c_ic = segment_sum(di, ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)

    return jnp.concatenate([c_pde, c_bc_val, c_bc_der, c_ic], axis=0)

@partial(jax.jit, static_argnames=("shapes",))
def C_and_J_cells_unscaled(theta, shapes,
                           X_f, ids_f,
                           X_bc_L, X_bc_R, ids_bc,
                           X_ic, u0_ic, ids_ic,
                           nu):
    def c_fun(th):
        return constraint_vector_cells_unscaled(
            th, shapes, X_f, ids_f, X_bc_L, X_bc_R, ids_bc, X_ic, u0_ic, ids_ic, nu
        )
    def c_fun_aux(th):
        c = c_fun(th)
        return c, c
    J_un, c_un = jax.jacrev(c_fun_aux, has_aux=True)(theta)
    return c_un, J_un

@partial(jax.jit, static_argnames=("shapes",))
def C_cells_unscaled(theta, shapes,
                     X_f, ids_f,
                     X_bc_L, X_bc_R, ids_bc,
                     X_ic, u0_ic, ids_ic,
                     nu):
    return constraint_vector_cells_unscaled(
        theta, shapes, X_f, ids_f, X_bc_L, X_bc_R, ids_bc, X_ic, u0_ic, ids_ic, nu
    )

# -----------------------------
# 10) KKT solve (LM proxy), lam_eff = lam*(s^2)
# -----------------------------
@jax.jit
def kkt_solve_once(H, Jac, Grad, Cons, lam_eff):
    n = H.shape[0]
    m = Cons.shape[0]
    I_m = jnp.eye(m, dtype=H.dtype)

    top = jnp.concatenate([H, Jac.T], axis=1)
    bottom = jnp.concatenate([Jac, -lam_eff * I_m], axis=1)
    KKT = jnp.concatenate([top, bottom], axis=0)

    rhs = -jnp.concatenate([Grad, Cons])
    sol = jnp.linalg.solve(KKT, rhs)
    d = sol[:n]
    y = sol[n:]
    return d, y

def cal_d_and_y(H, Jac, Grad, Cons, s_con,
               ridge_init=1e-6, eta_up=10.0, eta_down=0.25,
               lam_min=1e-9, lam_max=1e-4, res_tol=1e-6):
    lam = float(jnp.clip(DTYPE(ridge_init), DTYPE(lam_min), DTYPE(lam_max)))
    s2 = float(s_con * s_con)

    for _ in range(12):
        lam_eff = DTYPE(lam * s2)
        d, y = kkt_solve_once(H, Jac, Grad, Cons, lam_eff)

        res_vec = jnp.concatenate([H @ d + Jac.T @ y + Grad, Jac @ d + Cons])
        res = float(jnp.linalg.norm(res_vec, 2))

        if res <= res_tol:
            lam_next = float(jnp.clip(DTYPE(lam) * DTYPE(eta_down), DTYPE(lam_min), DTYPE(lam_max)))
            return d, y, lam_next, float(lam_eff), res

        lam = float(jnp.clip(DTYPE(lam) * DTYPE(eta_up), DTYPE(lam_min), DTYPE(lam_max)))

    lam_eff = DTYPE(lam * s2)
    d, y = kkt_solve_once(H, Jac, Grad, Cons, lam_eff)
    res_vec = jnp.concatenate([H @ d + Jac.T @ y + Grad, Jac @ d + Cons])
    res = float(jnp.linalg.norm(res_vec, 2))
    return d, y, lam, float(lam_eff), res

# -----------------------------
# 11) tau/ksi/alpha (your Zhou-style with damped KKT)
# -----------------------------
def cal_tau_mu(H, d, sigma, tau_pre, eps_tau, g, c, mu_eff, y):
    denom = float(g @ d + 0.5 * (d @ (H @ d)))
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(DTYPE(mu_eff) * y, 1))
    delta_c = c1 - mu_y1

    if (denom <= 1e-12) or (delta_c <= 0.0):
        tau_trial = float("inf")
    else:
        tau_trial = (1.0 - sigma) * delta_c / denom

    if tau_pre <= tau_trial:
        return tau_pre
    return min(tau_trial, (1.0 - eps_tau) * tau_pre)

def cal_ksi_mu(d, tau, ksi_old, eps_ksi, g, c, mu_eff, y):
    d2 = float(jnp.linalg.norm(d)**2)
    if d2 <= 1e-18 or tau <= 1e-18:
        return ksi_old

    gd = float(g @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(DTYPE(mu_eff) * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c

    ksi_trial = Dl / (tau * d2)
    ksi_trial = max(0.0, ksi_trial)

    if ksi_old <= ksi_trial:
        return ksi_old
    return min(ksi_trial, (1.0 - eps_ksi) * ksi_old)

def phi_mu(alpha, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma):
    gd = float(g @ d)
    d2 = float(d @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(DTYPE(mu_eff) * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c

    term1 = (eta - 1.0) * alpha * beta * Dl
    term2 = (abs(1.0 - alpha) - 1.0 + alpha) * c1
    term3 = 0.5 * (tau * L + Gamma) * (alpha ** 2) * d2
    return term1 + term2 + term3

def cal_alpha_mu(d, eta, beta, ksi, tau, L, Gamma, theta_val, g, c, mu_eff, y):
    denom = (tau * L + Gamma)
    if denom <= 1e-12:
        return 0.0

    alpha_min = 2.0 * (1.0 - eta) * beta * ksi * tau / denom
    a = max(alpha_min, 0.0)

    while (phi_mu(1.1 * a, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma) < 0.0
           and (1.1 * a < alpha_min + theta_val * beta)):
        a *= 1.1
    return float(a)

# -----------------------------
# 12) Training loop (HOLD + JITTER + EMA->FREEZE constraint scaling)
# -----------------------------
def train_sqp_cellwise_pointwise(theta0, shapes,
                                 X_data, u_data, x_grid, u0_grid, nu,
                                 x_min, x_max, t_min, t_max,
                                 # objective scaling (OFF)
                                 scale_obj=1.0,

                                 # --- constraint scaling controls ---
                                 con_scale_target=100.0,
                                 con_ema_alpha=0.05,
                                 con_warmup_iters=100,

                                 L=80.0, Gamma=80.0,
                                 max_iters=10000, print_every=10,
                                 beta0=1.0, beta_shift=200.0, beta_power=0.6,
                                 alpha_cap=1e1,
                                 seed=0,
                                 mc_con_batches=1,
                                 mc_obj_batches=1,
                                 obj_resample_each_mc=False,
                                 hold_data_K=10000,
                                 hold_con_K=10000,

                                 # ---- JITTER controls ----
                                 jitter_every_iter=True,
                                 jitter_kind="gaussian",
                                 jitter_sigma_x=1e-5,
                                 jitter_sigma_t=1e-5,
                                 jitter_clip_k=2.0,

                                 # ---- PDE objective controls ----
                                 w_pde_obj=10.0,
                                 hold_obj_K=10000,
                                 mc_pde_obj_jitter=16,
                                 jitter_obj_sigma_x=1e-5,
                                 jitter_obj_sigma_t=1e-5):

    key = random.PRNGKey(seed)
    theta = theta0
    n = theta.shape[0]
    H = jnp.eye(n, dtype=DTYPE)

    eta, sigma = 0.25, 0.1
    eps_tau, eps_ksi = 1e-2, 1e-2
    theta_val = 10.0
    tau_k, ksi_k = 5.0, 1.0
    lam_prev = 1e-6

    # constraint scaling state
    Jmag_ema = DTYPE(0.0)
    s_con = DTYPE(1.0)
    s_con_fixed = None

    # cached base samples (held)
    Xb = ub = None
    Xf = ids_f = None
    XL = XR = ids_bc = None
    Xic = u0ic = ids_ic = None
    X_obj_base = None

    # histories
    hist = {"pde_obj_mse": [], "feas_train": [], "feas_true": [], "total": []}
    rho_plot = 1.0

    H_beta = 10
    t0_wall = time.time()

    for k in range(1, max_iters + 1):
        # beta schedule (your block)
        k_beta = (k // H_beta) * H_beta
        beta_k = float(min(1.0, beta0 * (beta_shift / (beta_shift + k_beta)) ** beta_power))

        # ---- DATA sampling (held) ----
        if B_DATA > 0:
            if (k == 1) or ((k - 1) % int(hold_data_K) == 0) or (Xb is None):
                key, k_data = random.split(key, 2)
                Xb, ub = sample_data_batch(k_data, X_data, u_data, B_DATA)
        else:
            Xb, ub = None, None

        # ---- PDE objective base points (held) ----
        if (k == 1) or ((k - 1) % int(hold_obj_K) == 0) or (X_obj_base is None):
            key, k_objbase = random.split(key, 2)
            X_obj_base = sample_pde_obj_base_stratified(k_objbase, x_min, x_max, t_min, t_max)

        # ---- Objective MC: data (optional) + PDE objective ----
        S_obj = int(mc_obj_batches)
        obj_acc = DTYPE(0.0)
        g_acc = jnp.zeros((n,), dtype=DTYPE)

        for _ in range(S_obj):
            # data objective (optional)
            if B_DATA > 0:
                if obj_resample_each_mc:
                    key, k_data_i = random.split(key, 2)
                    Xb_i, ub_i = sample_data_batch(k_data_i, X_data, u_data, B_DATA)
                else:
                    Xb_i, ub_i = Xb, ub
                obj_d, g_d = F_and_g_data_scaled_batch(theta, shapes, Xb_i, ub_i, DTYPE(scale_obj))
            else:
                obj_d = DTYPE(0.0)
                g_d = jnp.zeros((n,), dtype=DTYPE)

            # PDE objective part (average over jitters)
            S_jit = int(mc_pde_obj_jitter)
            obj_r_acc = DTYPE(0.0)
            g_r_acc = jnp.zeros((n,), dtype=DTYPE)

            for _s in range(S_jit):
                key, kj = random.split(key, 2)
                X_obj_use = jitter_points(
                    kj, X_obj_base, x_min, x_max, t_min, t_max,
                    jitter_obj_sigma_x, jitter_obj_sigma_t,
                    kind=jitter_kind, clip_k=jitter_clip_k
                )
                obj_r_s, g_r_s = F_and_g_pde_obj(theta, shapes, X_obj_use, nu, w_pde_obj)
                obj_r_acc = obj_r_acc + obj_r_s
                g_r_acc = g_r_acc + g_r_s

            obj_r = obj_r_acc / DTYPE(S_jit)
            g_r   = g_r_acc   / DTYPE(S_jit)

            obj_i = obj_d + obj_r
            g_i   = g_d + g_r

            obj_acc = obj_acc + obj_i
            g_acc = g_acc + g_i

        obj_s = obj_acc / DTYPE(S_obj)
        g_s   = g_acc   / DTYPE(S_obj)

        # ---- BASE constraint sampling (held) ----
        if (k == 1) or ((k - 1) % int(hold_con_K) == 0) or (Xf is None):
            key, k_f, k_bc, k_ic = random.split(key, 4)
            Xf, ids_f = sample_pde_stratified(k_f, x_min, x_max, t_min, t_max)
            XL, XR, ids_bc = sample_bc_stratified(k_bc, x_min, x_max, t_min, t_max)
            Xic, u0ic, ids_ic = sample_ic_stratified(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

        # ---- JITTER every iteration ----
        if jitter_every_iter:
            key, kj1, kj2, kj3 = random.split(key, 4)

            Xf_use = jitter_points(kj1, Xf, x_min, x_max, t_min, t_max,
                                   jitter_sigma_x, jitter_sigma_t,
                                   kind=jitter_kind, clip_k=jitter_clip_k)

            XL_use = jitter_points(kj2, XL, x_min, x_max, t_min, t_max,
                                   0.0, jitter_sigma_t,
                                   kind=jitter_kind, clip_k=jitter_clip_k)
            XR_use = jitter_points(kj2, XR, x_min, x_max, t_min, t_max,
                                   0.0, jitter_sigma_t,
                                   kind=jitter_kind, clip_k=jitter_clip_k)

            Xic_use = jitter_points(kj3, Xic, x_min, x_max, t_min, t_max,
                                    jitter_sigma_x, 0.0,
                                    kind=jitter_kind, clip_k=jitter_clip_k)
            u0ic_use = jnp.interp(Xic_use[:, 0], x_grid, u0_grid)
        else:
            Xf_use, XL_use, XR_use, Xic_use, u0ic_use = Xf, XL, XR, Xic, u0ic

        # ---- Constraints/J MC average ----
        S_con = int(mc_con_batches)
        c_acc = jnp.zeros((M_CON,), dtype=DTYPE)
        J_acc = jnp.zeros((M_CON, n), dtype=DTYPE)

        for _ in range(S_con):
            c_tmp, J_tmp = C_and_J_cells_unscaled(
                theta, shapes,
                Xf_use, ids_f,
                XL_use, XR_use, ids_bc,
                Xic_use, u0ic_use, ids_ic,
                nu
            )
            c_acc = c_acc + c_tmp
            J_acc = J_acc + J_tmp

        c_un = c_acc / DTYPE(S_con)
        J_un = J_acc / DTYPE(S_con)

        # -----------------------------
        # ✅ YOUR REQUEST: EMA scaling for 100 iters then FREEZE
        # -----------------------------
        mag = jnp.sqrt(jnp.mean(J_un**2) + DTYPE(EPS))  # scalar RMS magnitude of J

        if s_con_fixed is None:
            # EMA
            if Jmag_ema == 0:
                Jmag_ema = mag
            else:
                a = DTYPE(con_ema_alpha)
                Jmag_ema = (DTYPE(1.0) - a) * Jmag_ema + a * mag

            # shrink-only scale (<=1)
            s_con = DTYPE(con_scale_target) / jnp.maximum(DTYPE(con_scale_target), Jmag_ema + DTYPE(EPS))

            # freeze after warmup
            if k >= int(con_warmup_iters):
                s_con_fixed = s_con
        else:
            s_con = s_con_fixed

        Cons = s_con * c_un
        Jac  = s_con * J_un

        # ---- KKT direction ----
        d, y, lam_prev, mu_eff, kkt_res = cal_d_and_y(
            H, Jac, g_s, Cons, s_con, ridge_init=lam_prev
        )

        # ---- Step size ----
        dn = float(jnp.linalg.norm(d, jnp.inf))
        if dn > 1e-10:
            tau_k = cal_tau_mu(H, d, sigma, tau_k, eps_tau, g_s, Cons, mu_eff, y)
            ksi_k = cal_ksi_mu(d, tau_k, ksi_k, eps_ksi, g_s, Cons, mu_eff, y)
            alpha = cal_alpha_mu(d, eta, beta_k, ksi_k, tau_k, L, Gamma, theta_val, g_s, Cons, mu_eff, y)
        else:
            alpha = 2.0 * (1.0 - eta) * beta_k * ksi_k * tau_k / (tau_k * L + Gamma)

        alpha = float(min(alpha, alpha_cap))
        theta = theta + DTYPE(alpha) * d

        # ---- Diagnostics (after update) ----
        # PDE objective MSE (avg over jitters)
        pde_obj_mse_acc = DTYPE(0.0)
        for _s in range(int(mc_pde_obj_jitter)):
            key, kj_dbg = random.split(key, 2)
            X_obj_dbg = jitter_points(
                kj_dbg, X_obj_base, x_min, x_max, t_min, t_max,
                jitter_obj_sigma_x, jitter_obj_sigma_t,
                kind=jitter_kind, clip_k=jitter_clip_k
            )
            pde_obj_mse_acc = pde_obj_mse_acc + pde_mse_on_points(theta, shapes, X_obj_dbg, nu)
        pde_obj_mse = float(pde_obj_mse_acc / DTYPE(int(mc_pde_obj_jitter)))

        # training feasibility (on jittered points)
        c_train_un = C_cells_unscaled(
            theta, shapes, Xf_use, ids_f, XL_use, XR_use, ids_bc, Xic_use, u0ic_use, ids_ic, nu
        )
        feas_train_unscaled = float(jnp.mean(c_train_un**2))
        feas_train_scaled   = float(jnp.mean((s_con * c_train_un)**2))

        # "true" feasibility (on held BASE points, no new jitter)
        c_base_un = C_cells_unscaled(
            theta, shapes, Xf, ids_f, XL, XR, ids_bc, Xic, u0ic, ids_ic, nu
        )
        feas_true_unscaled = float(jnp.mean(c_base_un**2))
        feas_true_scaled   = float(jnp.mean((s_con * c_base_un)**2))

        total = pde_obj_mse + rho_plot * feas_train_unscaled

        hist["pde_obj_mse"].append(pde_obj_mse)
        hist["feas_train"].append(feas_train_unscaled)
        hist["feas_true"].append(feas_true_unscaled)
        hist["total"].append(total)

        if k % int(print_every) == 0:
            station = float(jnp.linalg.norm(g_s + Jac.T @ y, jnp.inf))
            d_inf = float(jnp.linalg.norm(d, jnp.inf))
            d_l2  = float(jnp.linalg.norm(d, 2))

            if B_DATA > 0:
                params_now = unflatten_params(theta, shapes)
                data_mse_true = float(data_mse_unscaled(params_now, Xb, ub))
            else:
                data_mse_true = float("nan")

            print(
                f"[iter={k}] obj_s={float(obj_s):.3e} PDE_obj_MSE={pde_obj_mse:.3e} "
                f"data_lossmse={data_mse_true:.3e} "
                f"feas_true_un={feas_true_unscaled:.3e} feas_train_un={feas_train_unscaled:.3e} "
                f"station={station:.3e} tau={tau_k:.2e} "
                f"alpha={alpha:.2e} beta={beta_k:.2e} "
                f"lam={lam_prev:.2e} kkt_res={kkt_res:.1e} "
                f"s_con={float(s_con):.2e} (frozen={s_con_fixed is not None}) "
                f"| ||d||_inf={d_inf:.2e} ||d||_l2={d_l2:.2e}"
            )

    print(f"[done] elapsed={time.time() - t0_wall:.2f}s")
    return theta, hist

# -----------------------------
# 13) Eval helpers
# -----------------------------
def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape

@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]

def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.array(X_grid_np, dtype=DTYPE)
    u_pred = np.array(predict_u(theta, shapes, X_grid)).reshape(grid_shape)
    u_true = usol_np
    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))
    return mse, rel_l2, u_pred

def plot_heatmaps(x_np, t_np, u_true, u_pred):
    import matplotlib.pyplot as plt
    err = u_pred - u_true
    plt.figure(); plt.pcolormesh(x_np, t_np, u_true, shading="auto"); plt.colorbar(); plt.title("True Solution"); plt.xlabel("x"); plt.ylabel("t"); plt.show()
    plt.figure(); plt.pcolormesh(x_np, t_np, u_pred, shading="auto"); plt.colorbar(); plt.title("sto-SQP Prediction"); plt.xlabel("x"); plt.ylabel("t"); plt.show()
    plt.figure(); plt.pcolormesh(x_np, t_np, err, shading="auto"); plt.colorbar(); plt.title("Absolute Error"); plt.xlabel("x"); plt.ylabel("t"); plt.show()

# -----------------------------
# 14) main
# -----------------------------
def main():
    t_np, x_np, usol_np, nu = load_burgers_mat("data/burgers.mat")
    print("loaded burgers.mat:", "t", t_np.shape, "x", x_np.shape, "usol", usol_np.shape, "nu", nu)

    x_min, x_max = float(x_np.min()), float(x_np.max())
    t_min, t_max = float(t_np.min()), float(t_np.max())
    print(f"Domain: x in [{x_min}, {x_max}], t in [{t_min}, {t_max}]")

    X_data_np, u_data_np = build_full_data_points(t_np, x_np, usol_np)
    X_data = jnp.array(X_data_np, dtype=DTYPE)
    u_data = jnp.array(u_data_np, dtype=DTYPE)

    x_grid = jnp.array(x_np, dtype=DTYPE)
    u0_grid = jnp.array(usol_np[0, :], dtype=DTYPE)

    # network
    hidden_dim = 30
    num_hidden = 3
    layer_sizes = [2] + [hidden_dim] * num_hidden + [1]

    key = random.PRNGKey(0)
    key, key_params = random.split(key)
    params0 = init_mlp_params(key_params, layer_sizes)
    theta0, shapes = flatten_params(params0)

    # ✅ DO NOT SCALE OBJECTIVE
    scale_obj = DTYPE(1.0)

    print("scale_obj (OFF) =", float(scale_obj))
    print("Constraint dimension M_CON =", M_CON)

    theta_star, hist_sqp = train_sqp_cellwise_pointwise(
        theta0, shapes,
        X_data, u_data, x_grid, u0_grid, nu,
        x_min=x_min, x_max=x_max, t_min=t_min, t_max=t_max,

        scale_obj=scale_obj,

        # ✅ EMA scaling for 100 iters then freeze
        con_scale_target=100.0,
        con_ema_alpha=0.05,
        con_warmup_iters=100,

        L=80.0, Gamma=80.0,
        max_iters=10000, print_every=10,
        beta0=1.0, beta_shift=200.0, beta_power=0.6,
        alpha_cap=1e1,
        seed=0,

        mc_con_batches=1,
        mc_obj_batches=1,
        obj_resample_each_mc=False,

        hold_data_K=10000,
        hold_con_K=10000,

        # jitter for constraints
        jitter_every_iter=True,
        jitter_kind="gaussian",
        jitter_sigma_x=1e-5,
        jitter_sigma_t=1e-5,
        jitter_clip_k=2.0,

        # PDE objective
        w_pde_obj=10,
        hold_obj_K=10000,
        mc_pde_obj_jitter=16,
        jitter_obj_sigma_x=1e-5,
        jitter_obj_sigma_t=1e-5,
    )

    np.save("burgers_rs.npy", np.array(theta_star))

    np.save("hist_sqp_total.npy",      np.array(hist_sqp["total"]))
    np.save("hist_sqp_pdeobj.npy",     np.array(hist_sqp["pde_obj_mse"]))
    np.save("hist_sqp_feas_train.npy", np.array(hist_sqp["feas_train"]))
    np.save("hist_sqp_feas_true.npy",  np.array(hist_sqp["feas_true"]))
    print("Saved theta + histories")

    mse, rel_l2, u_pred = eval_full_grid(theta_star, shapes, x_np, t_np, usol_np)
    print(f"[EVAL] full-grid MSE={mse:.3e}, relL2={rel_l2:.3e}")

    plot_heatmaps(x_np, t_np, usol_np, u_pred)

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'data/burgers.mat'